# Extração SQL Server → MinIO (landing-zone)

Descobre tabelas via `INFORMATION_SCHEMA`, exporta cada uma para CSV em `s3a://landing-zone/<schema>_<tabela>/` usando PySpark + S3A (MinIO).

Após iniciar o Spark, uma célula garante o bucket **`landing-zone`** no MinIO (API S3 com **boto3**), alinhada ao serviço `minio-init` do Docker quando este não tiver corrido.

In [18]:
import os
import re
from pathlib import Path

from dotenv import load_dotenv

# Carrega .env a partir da raiz do projeto (pai da pasta notebook/)
_root = Path.cwd()
if _root.name == "notebook":
    _root = _root.parent
load_dotenv(_root / ".env")

required_minio = ["MINIO_ROOT_USER", "MINIO_ROOT_PASSWORD", "MINIO_S3_ENDPOINT"]
missing = [k for k in required_minio if not os.getenv(k)]
if missing:
    raise RuntimeError(f"Defina no .env: {', '.join(missing)}")

jdbc_url_env = os.getenv("MSSQL_JDBC_URL", "").strip()
if jdbc_url_env:
    JDBC_URL = jdbc_url_env
else:
    host = os.getenv("MSSQL_JDBC_HOST", "localhost")
    port = os.getenv("MSSQL_JDBC_PORT", "1433")
    database = os.getenv("MSSQL_JDBC_DATABASE", "master")
    enc = os.getenv("MSSQL_JDBC_ENCRYPT", "false")
    trust = os.getenv("MSSQL_JDBC_TRUST_SERVER_CERTIFICATE", "true")
    JDBC_URL = (
        f"jdbc:sqlserver://{host}:{port};databaseName={database};"
        f"encrypt={enc};trustServerCertificate={trust}"
    )

JDBC_USER = os.getenv("MSSQL_JDBC_USER", "sa")
JDBC_PASSWORD = os.getenv("MSSQL_JDBC_PASSWORD") or os.getenv("MSSQL_SA_PASSWORD")
if not JDBC_PASSWORD:
    raise RuntimeError("Defina MSSQL_JDBC_PASSWORD ou MSSQL_SA_PASSWORD no .env")

MINIO_ENDPOINT = os.getenv("MINIO_S3_ENDPOINT")
MINIO_ACCESS_KEY = os.getenv("MINIO_ROOT_USER")
MINIO_SECRET_KEY = os.getenv("MINIO_ROOT_PASSWORD")

LANDING_BUCKET = "landing-zone"

In [19]:
from pyspark.sql import SparkSession

# Pacotes compatíveis com Spark 3.5.x (Hadoop 3.3.4 interno)
packages = (
    "com.microsoft.sqlserver:mssql-jdbc:12.8.1.jre11,"
    "io.delta:delta-spark_2.12:3.2.0,"
    "org.apache.hadoop:hadoop-aws:3.3.4,"
    "com.amazonaws:aws-java-sdk-bundle:1.12.262"
)

spark = (
    SparkSession.builder.appName("extracao-sqlserver-landing-zone")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config(
        "spark.sql.catalog.spark_catalog",
        "org.apache.spark.sql.delta.catalog.DeltaCatalog",
    )
    .config("spark.jars.packages", packages)
    # MinIO via S3A
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.access.key", MINIO_ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", MINIO_SECRET_KEY)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark inicializado:", spark.version)

Spark inicializado: 3.5.3


### Bucket MinIO (criação idempotente)

Garante o bucket **`landing-zone`** antes das escritas CSV (API S3 com **boto3**), em linha com o **`minio-init`** do Docker.

In [20]:
import boto3
from botocore.exceptions import ClientError


def ensure_minio_bucket(endpoint: str, access_key: str, secret_key: str, bucket: str) -> None:
    client = boto3.client(
        "s3",
        endpoint_url=endpoint,
        aws_access_key_id=access_key,
        aws_secret_access_key=secret_key,
    )
    try:
        client.create_bucket(Bucket=bucket)
    except ClientError as exc:
        code = exc.response.get("Error", {}).get("Code", "")
        if code not in ("BucketAlreadyOwnedByYou", "BucketAlreadyExists"):
            raise


ensure_minio_bucket(MINIO_ENDPOINT, MINIO_ACCESS_KEY, MINIO_SECRET_KEY, LANDING_BUCKET)
print("Bucket MinIO:", LANDING_BUCKET)

Bucket MinIO: landing-zone


In [21]:
jdbc_props = {
    "user": JDBC_USER,
    "password": JDBC_PASSWORD,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
}

tables_subquery = (
    "(SELECT TABLE_SCHEMA, TABLE_NAME FROM INFORMATION_SCHEMA.TABLES "
    "WHERE TABLE_TYPE = 'BASE TABLE' "
    "AND TABLE_SCHEMA NOT IN ('sys', 'INFORMATION_SCHEMA')) AS user_tables"
)

tables_df = spark.read.jdbc(url=JDBC_URL, table=tables_subquery, properties=jdbc_props)
table_rows = tables_df.collect()
print(f"Tabelas encontradas: {len(table_rows)}")
for r in table_rows:
    print(f"  - [{r.TABLE_SCHEMA}].[{r.TABLE_NAME}]")

Tabelas encontradas: 6
  - [dbo].[Categoria]
  - [dbo].[Autor]
  - [dbo].[Livro]
  - [dbo].[Membro]
  - [dbo].[Emprestimo]
  - [dbo].[Multa]


In [22]:
# Verificar o que existe no SQL Server
query_bancos = "(SELECT name FROM sys.databases) AS bancos"

bancos = spark.read.jdbc(url=JDBC_URL, table=query_bancos, properties=jdbc_props).collect()

print("Bancos existentes:")
for b in bancos:
    print(f"  → {b.name}")

Bancos existentes:
  → master
  → tempdb
  → model
  → msdb
  → BibliotecaDb


In [23]:
# Verificar tabelas dentro do BibliotecaDb
query_tabelas = """
    (
        SELECT TABLE_SCHEMA, TABLE_NAME
        FROM BibliotecaDb.INFORMATION_SCHEMA.TABLES
        WHERE TABLE_TYPE = 'BASE TABLE'
    ) AS tabelas
"""

tabelas = spark.read.jdbc(url=JDBC_URL, table=query_tabelas, properties=jdbc_props).collect()

print(f"Tabelas encontradas no BibliotecaDb: {len(tabelas)}")
for t in tabelas:
    print(f"  → [{t.TABLE_SCHEMA}].[{t.TABLE_NAME}]")

Tabelas encontradas no BibliotecaDb: 6
  → [dbo].[Categoria]
  → [dbo].[Autor]
  → [dbo].[Livro]
  → [dbo].[Membro]
  → [dbo].[Emprestimo]
  → [dbo].[Multa]


In [24]:
def sanitize_folder_name(schema: str, table: str) -> str:
    raw = f"{schema}_{table}"
    safe = re.sub(r"[^a-zA-Z0-9_-]+", "_", raw)
    return safe.strip("_") or "table"


# Query corrigida apontando direto para BibliotecaDb
query_tabelas = """
    (
        SELECT TABLE_SCHEMA, TABLE_NAME
        FROM BibliotecaDb.INFORMATION_SCHEMA.TABLES
        WHERE TABLE_TYPE = 'BASE TABLE'
    ) AS tabelas
"""

table_rows = spark.read.jdbc(url=JDBC_URL, table=query_tabelas, properties=jdbc_props).collect()

success = []
errors = []
total_rows = 0

for row in table_rows:
    schema, table = row.TABLE_SCHEMA, row.TABLE_NAME
    fq = f"[BibliotecaDb].[{schema}].[{table}]"
    folder = sanitize_folder_name(schema, table)
    target = f"s3a://{LANDING_BUCKET}/{folder}/"
    print(f"Extraindo {fq} → {target}")
    try:
        df = spark.read.jdbc(url=JDBC_URL, table=fq, properties=jdbc_props)
        n = df.count()
        total_rows += n
        (
            df.write.mode("overwrite")
            .option("header", True)
            .csv(target)
        )
        success.append({"schema": schema, "table": table, "rows": n, "path": target})
        print(f"  OK — {n} linhas gravadas")
    except Exception as exc:
        errors.append({"schema": schema, "table": table, "error": str(exc)})
        print(f"  ERRO — {exc}")

print(f"\n=== Resumo ===")
print(f"Sucesso: {len(success)}")
print(f"Com erro: {len(errors)}")
print(f"Total de linhas extraídas: {total_rows}")

Extraindo [BibliotecaDb].[dbo].[Categoria] → s3a://landing-zone/dbo_Categoria/


26/05/06 20:30:49 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/05/06 20:30:49 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/05/06 20:30:50 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


  OK — 5 linhas gravadas
Extraindo [BibliotecaDb].[dbo].[Autor] → s3a://landing-zone/dbo_Autor/


26/05/06 20:30:51 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/05/06 20:30:51 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


  OK — 10 linhas gravadas
Extraindo [BibliotecaDb].[dbo].[Livro] → s3a://landing-zone/dbo_Livro/


...

  OK — 15 linhas gravadas
Extraindo [BibliotecaDb].[dbo].[Emprestimo] → s3a://landing-zone/dbo_Emprestimo/


26/05/06 20:30:53 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/05/06 20:30:54 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


  OK — 30 linhas gravadas
Extraindo [BibliotecaDb].[dbo].[Multa] → s3a://landing-zone/dbo_Multa/


26/05/06 20:30:54 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/05/06 20:30:54 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


  OK — 8 linhas gravadas

=== Resumo ===
Sucesso: 6
Com erro: 0
Total de linhas extraídas: 88


In [25]:
print("\n=== Resumo ===")
print("Sucesso:", len(success))
print("Com erro:", len(errors))
print("Total de linhas extraídas:", total_rows)
if errors:
    print("\nFalhas:")
    for e in errors:
        print(f"  [{e['schema']}].[{e['table']}]: {e['error']}")


=== Resumo ===
Sucesso: 6
Com erro: 0
Total de linhas extraídas: 88


In [26]:
def list_s3a_prefix(spark_session, uri: str, indent: str = "") -> None:
    """Lista caminhos sob um prefixo s3a usando a API Hadoop já configurada no Spark."""
    jvm = spark_session.sparkContext._jvm
    hadoop_conf = spark_session.sparkContext._jsc.hadoopConfiguration()
    path = jvm.org.apache.hadoop.fs.Path(uri)
    fs = path.getFileSystem(hadoop_conf)
    if not fs.exists(path):
        print(f"{indent}(prefixo inexistente ou vazio)")
        return
    statuses = fs.listStatus(path)
    for st in statuses:
        p = st.getPath().toString()
        if st.isDirectory():
            print(f"{indent}{p}/")
            list_s3a_prefix(spark_session, p + "/", indent + "  ")
        else:
            print(f"{indent}{p}")


print(f"Objetos em s3a://{LANDING_BUCKET}/")
list_s3a_prefix(spark, f"s3a://{LANDING_BUCKET}/")

Objetos em s3a://landing-zone/
s3a://landing-zone/dbo_Autor/
  s3a://landing-zone/dbo_Autor/_SUCCESS
  s3a://landing-zone/dbo_Autor/part-00000-71e7959e-101d-4c24-82a4-4e0182579f6d-c000.csv
s3a://landing-zone/dbo_Categoria/
  s3a://landing-zone/dbo_Categoria/_SUCCESS
  s3a://landing-zone/dbo_Categoria/part-00000-a2e5bf34-7b1e-4487-809e-66f17c053005-c000.csv
s3a://landing-zone/dbo_Emprestimo/
  s3a://landing-zone/dbo_Emprestimo/_SUCCESS
  s3a://landing-zone/dbo_Emprestimo/part-00000-262a81c2-8cf9-44ce-b79e-3dd08310cae9-c000.csv
s3a://landing-zone/dbo_Livro/
  s3a://landing-zone/dbo_Livro/_SUCCESS
  s3a://landing-zone/dbo_Livro/part-00000-d46b9a89-a9db-421e-8a2b-4e2280e01cf0-c000.csv
s3a://landing-zone/dbo_Membro/
  s3a://landing-zone/dbo_Membro/_SUCCESS
  s3a://landing-zone/dbo_Membro/part-00000-2e253711-f3d1-4ada-aa49-da0cb7832005-c000.csv
s3a://landing-zone/dbo_Multa/
  s3a://landing-zone/dbo_Multa/_SUCCESS
  s3a://landing-zone/dbo_Multa/part-00000-88421f36-fd1a-45c7-a6e3-72669b1e6a58-

In [27]:
spark.stop()